# RecA Classical Machine Learning Modeling Workflow

**Publication-ready notebook** for constructing, optimizing, evaluating, and interpreting classical machine-learning models for **RecA inhibitor classification** using PaDEL molecular fingerprints/descriptors.

This notebook combines:

1. The RecA-specific full modeling workflow from the user's notebook.
2. The clearer SVM/RF/XGBoost modeling structure from the senior reference notebook.

The workflow is designed to be reproducible and thesis/manuscript-ready: stratified splitting, leakage-aware preprocessing, cross-validation, hyperparameter optimization, independent test-set evaluation, ROC/confusion-matrix visualization, model comparison, feature importance, optional SHAP interpretation, and export of all results.

## 1. Environment setup

Run this cell first. The notebook uses common scientific Python packages. Optional packages such as `xgboost` and `shap` are handled safely: if they are unavailable, the notebook will continue with the available models.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    make_scorer,
)
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception:
    XGBClassifier = None
    HAS_XGBOOST = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    shap = None
    HAS_SHAP = False

RANDOM_STATE = 42
TEST_SIZE = 0.20
N_SPLITS = 10

PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs" / "modeling_publication_ready"
FIGURE_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"

for folder in [OUTPUT_DIR, FIGURE_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("XGBoost available:", HAS_XGBOOST)
print("SHAP available:", HAS_SHAP)
print("Output directory:", OUTPUT_DIR.resolve())

## 2. Input dataset

Recommended input from the previous workflow:

- `outputs/feature_selection/03_recA_top_200_fscore.csv`, or
- `outputs/02_recA_modeling_matrix.csv`, or
- any RecA fingerprint/feature-selection matrix containing descriptor columns and a class label.

The target column can be named `class`, `bioactivity_class`, `Activity`, `label`, or `target`. The notebook automatically detects the target column and converts common active/inactive labels to binary values.

In [ ]:
# Change this path if your dataset has a different file name.
CANDIDATE_INPUT_FILES = [
    PROJECT_DIR / "outputs" / "feature_selection" / "03_recA_top_200_fscore.csv",
    PROJECT_DIR / "outputs" / "feature_selection" / "03_recA_low_variance_filtered.csv",
    PROJECT_DIR / "outputs" / "02_recA_modeling_matrix.csv",
    PROJECT_DIR / "02_recA_modeling_matrix.csv",
    PROJECT_DIR / "03_recA_top_200_fscore.csv",
]

INPUT_FILE = next((p for p in CANDIDATE_INPUT_FILES if p.exists()), None)

if INPUT_FILE is None:
    raise FileNotFoundError(
        "No input dataset was found. Please place the RecA modeling matrix in one of these locations:\n"
        + "\n".join(str(p) for p in CANDIDATE_INPUT_FILES)
    )

print("Using input file:", INPUT_FILE)
df = pd.read_csv(INPUT_FILE)
print("Dataset shape:", df.shape)
df.head()

## 3. Data cleaning and target preparation

This section removes metadata columns from the feature matrix, keeps only numeric descriptors/fingerprints, removes missing/infinite values, and prepares the binary classification target.

In [ ]:
TARGET_CANDIDATES = ["class", "bioactivity_class", "Activity", "activity", "label", "target", "y"]
META_COLUMNS = [
    "Name", "Molecule ChEMBL ID", "chembl_id", "canonical_smiles", "SMILES", "smiles",
    "standard_type", "standard_value", "standard_units", "pEC50", "EC50", "IC50",
    "assay_description", "target_organism", "target_pref_name"
]


def detect_target_column(data: pd.DataFrame) -> str:
    for col in TARGET_CANDIDATES:
        if col in data.columns:
            return col
    raise ValueError(f"Target column not found. Expected one of: {TARGET_CANDIDATES}")


def encode_binary_target(y: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(y):
        unique_values = sorted(pd.Series(y.dropna().unique()).tolist())
        if set(unique_values).issubset({0, 1}):
            return y.astype(int)
        # For continuous activity values, use median split only as a fallback.
        threshold = y.median()
        print(f"Numeric non-binary target detected. Applying median split at {threshold:.4f}.")
        return (y >= threshold).astype(int)

    mapping = {
        "active": 1, "actives": 1, "1": 1, "yes": 1, "high": 1,
        "inactive": 0, "inactives": 0, "0": 0, "no": 0, "low": 0,
    }
    y_clean = y.astype(str).str.strip().str.lower().map(mapping)
    if y_clean.isna().any():
        unknown = sorted(y.astype(str)[y_clean.isna()].unique().tolist())
        raise ValueError(f"Unrecognized target labels: {unknown}")
    return y_clean.astype(int)


target_col = detect_target_column(df)
y = encode_binary_target(df[target_col])

feature_df = df.drop(columns=[target_col], errors="ignore")
feature_df = feature_df.drop(columns=[c for c in META_COLUMNS if c in feature_df.columns], errors="ignore")
feature_df = feature_df.select_dtypes(include=[np.number]).replace([np.inf, -np.inf], np.nan)
feature_df = feature_df.dropna(axis=1, how="all").fillna(feature_df.median(numeric_only=True))

# Remove constant descriptors before splitting. This is safe because it does not use the target.
constant_cols = [c for c in feature_df.columns if feature_df[c].nunique(dropna=False) <= 1]
X = feature_df.drop(columns=constant_cols, errors="ignore")

print("Detected target column:", target_col)
print("Removed constant columns:", len(constant_cols))
print("Final feature matrix:", X.shape)
print("Class distribution:")
print(y.value_counts().rename({0: "inactive", 1: "active"}))

if y.nunique() != 2:
    raise ValueError("This notebook requires a binary classification target.")

## 4. Stratified train-test split

The train-test split follows the senior notebook style but uses RecA data and keeps class proportions balanced using `stratify=y`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train distribution:\n", y_train.value_counts(normalize=True).round(3))
print("y_test distribution:\n", y_test.value_counts(normalize=True).round(3))

## 5. Evaluation functions

Metrics reported for manuscript/thesis tables:

- Accuracy
- Precision
- Recall/Sensitivity
- F1-score
- Matthews correlation coefficient (MCC)
- ROC-AUC

In [ ]:
def evaluate_predictions(y_true, y_pred, y_proba=None) -> dict:
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1_score": f1_score(y_true, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_true, y_pred),
    }
    if y_proba is not None and len(np.unique(y_true)) == 2:
        metrics["ROC_AUC"] = roc_auc_score(y_true, y_proba)
    else:
        metrics["ROC_AUC"] = np.nan
    return metrics


def evaluate_cross_validation(estimator, X_data, y_data, cv) -> dict:
    scoring = {
        "Accuracy": "accuracy",
        "Precision": "precision",
        "Recall": "recall",
        "F1_score": "f1",
        "MCC": make_scorer(matthews_corrcoef),
        "ROC_AUC": "roc_auc",
    }
    cv_results = cross_validate(estimator, X_data, y_data, cv=cv, scoring=scoring, n_jobs=-1)
    return {
        metric: float(np.mean(cv_results[f"test_{metric}"]))
        for metric in scoring.keys()
    }


def metrics_to_frame(metrics: dict, model_name: str, dataset_name: str) -> pd.DataFrame:
    row = {"Model": model_name, "Dataset": dataset_name}
    row.update(metrics)
    return pd.DataFrame([row])


def get_positive_probability(model, X_data):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_data)[:, 1]
    if hasattr(model, "decision_function"):
        scores = model.decision_function(X_data)
        return (scores - scores.min()) / (scores.max() - scores.min() + 1e-12)
    return None

## 6. Model pipelines and hyperparameter grids

The models follow the senior notebook structure: **SVM**, **Random Forest**, and **XGBoost**. ExtraTrees is also included as a strong tree-based benchmark commonly useful for sparse molecular fingerprint data.

To reduce data leakage, preprocessing and feature selection are placed inside the `Pipeline`, so cross-validation learns them only from the training folds.

In [ ]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# K is capped to avoid errors when the number of descriptors is smaller than 200.
K_BEST = min(200, X_train.shape[1])

model_spaces = {
    "SVM": {
        "pipeline": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("select", SelectKBest(score_func=f_classif, k=K_BEST)),
            ("scaler", StandardScaler()),
            ("model", SVC(probability=True, random_state=RANDOM_STATE)),
        ]),
        "params": {
            "model__C": [0.1, 1, 10, 100],
            "model__gamma": ["scale", 0.01, 0.1, 1],
            "model__kernel": ["rbf", "linear"],
        },
    },
    "RandomForest": {
        "pipeline": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("select", SelectKBest(score_func=f_classif, k=K_BEST)),
            ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")),
        ]),
        "params": {
            "model__n_estimators": [200, 500],
            "model__max_depth": [None, 5, 10, 15],
            "model__min_samples_split": [2, 4],
            "model__min_samples_leaf": [1, 2],
        },
    },
    "ExtraTrees": {
        "pipeline": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("select", SelectKBest(score_func=f_classif, k=K_BEST)),
            ("model", ExtraTreesClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced")),
        ]),
        "params": {
            "model__n_estimators": [300, 500, 800],
            "model__max_depth": [None, 10, 20],
            "model__min_samples_split": [2, 4],
            "model__min_samples_leaf": [1, 2],
            "model__max_features": ["sqrt", "log2", None],
        },
    },
}

if HAS_XGBOOST:
    model_spaces["XGBoost"] = {
        "pipeline": Pipeline([
            ("variance", VarianceThreshold(threshold=0.0)),
            ("select", SelectKBest(score_func=f_classif, k=K_BEST)),
            ("model", XGBClassifier(
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ]),
        "params": {
            "model__n_estimators": [100, 300, 500],
            "model__max_depth": [3, 4, 5],
            "model__learning_rate": [0.01, 0.05, 0.1],
            "model__subsample": [0.8, 1.0],
            "model__colsample_bytree": [0.8, 1.0],
        },
    }

print("Models to benchmark:", list(model_spaces.keys()))

## 7. Hyperparameter optimization and model evaluation

Each model is optimized using stratified 10-fold cross-validation on the training set only. The independent test set is used once for final evaluation.

In [ ]:
best_models = {}
all_metrics = []
best_params = {}

for model_name, spec in model_spaces.items():
    print(f"\n===== {model_name} =====")
    search = GridSearchCV(
        estimator=spec["pipeline"],
        param_grid=spec["params"],
        scoring="roc_auc",
        cv=cv,
        n_jobs=-1,
        refit=True,
        verbose=0,
    )
    search.fit(X_train, y_train)
    best_model = search.best_estimator_
    best_models[model_name] = best_model
    best_params[model_name] = search.best_params_

    cv_metrics = evaluate_cross_validation(best_model, X_train, y_train, cv=cv)
    y_pred = best_model.predict(X_test)
    y_proba = get_positive_probability(best_model, X_test)
    test_metrics = evaluate_predictions(y_test, y_pred, y_proba)

    all_metrics.append(metrics_to_frame(cv_metrics, model_name, "Train_10fold_CV"))
    all_metrics.append(metrics_to_frame(test_metrics, model_name, "Independent_Test"))

    print("Best parameters:", search.best_params_)
    print("Best CV ROC-AUC from GridSearch:", round(search.best_score_, 4))
    print("Independent test ROC-AUC:", round(test_metrics["ROC_AUC"], 4))

metrics_df = pd.concat(all_metrics, ignore_index=True)
metrics_df = metrics_df.sort_values(["Dataset", "ROC_AUC"], ascending=[True, False])
metrics_df.to_csv(OUTPUT_DIR / "04_recA_model_comparison_metrics.csv", index=False)

with open(OUTPUT_DIR / "04_recA_best_hyperparameters.json", "w") as f:
    json.dump(best_params, f, indent=2)

metrics_df.round(4)

## 8. Select the best model

The best model is selected based on independent test ROC-AUC. For small molecular datasets, also inspect MCC, F1-score, and the confusion matrix before making a final manuscript claim.

In [ ]:
test_table = metrics_df[metrics_df["Dataset"] == "Independent_Test"].copy()
best_model_name = test_table.sort_values("ROC_AUC", ascending=False).iloc[0]["Model"]
best_model = best_models[best_model_name]

print("Best model based on independent test ROC-AUC:", best_model_name)
display(test_table.sort_values("ROC_AUC", ascending=False).round(4))

joblib.dump(best_model, MODEL_DIR / f"04_recA_best_model_{best_model_name}.joblib")
print("Saved best model to:", MODEL_DIR / f"04_recA_best_model_{best_model_name}.joblib")

## 9. Confusion matrices

This section saves publication-ready confusion matrices for all models.

In [ ]:
for model_name, model in best_models.items():
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(5, 4), dpi=300)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Inactive", "Active"])
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"{model_name} Confusion Matrix")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"04_confusion_matrix_{model_name}.png", bbox_inches="tight")
    plt.show()

## 10. ROC curves

ROC curves are generated for all optimized models and saved as high-resolution PNG files.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=300)

roc_rows = []
for model_name, model in best_models.items():
    y_proba = get_positive_probability(model, X_test)
    if y_proba is None:
        continue
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    auc_value = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f"{model_name} (AUC = {auc_value:.3f})")
    roc_rows.append(pd.DataFrame({"Model": model_name, "FPR": fpr, "TPR": tpr, "Threshold": thresholds}))

ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves for RecA Classification Models")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_roc_curves_all_models.png", bbox_inches="tight")
plt.show()

if roc_rows:
    pd.concat(roc_rows, ignore_index=True).to_csv(OUTPUT_DIR / "04_recA_roc_curve_coordinates.csv", index=False)

## 11. Model comparison table

This table can be copied directly into the thesis/manuscript after checking formatting.

In [ ]:
comparison_pivot = metrics_df.pivot(index="Model", columns="Dataset", values=["Accuracy", "Precision", "Recall", "F1_score", "MCC", "ROC_AUC"])
comparison_pivot.to_csv(OUTPUT_DIR / "04_recA_model_comparison_pivot.csv")
comparison_pivot.round(4)

## 12. Selected features from the best pipeline

Because feature selection is inside the pipeline, this cell retrieves the actual selected descriptors/fingerprints used by the best model.

In [ ]:
def extract_selected_feature_names(pipeline: Pipeline, original_columns: pd.Index) -> list[str]:
    columns = np.array(original_columns)

    if "variance" in pipeline.named_steps:
        variance_mask = pipeline.named_steps["variance"].get_support()
        columns = columns[variance_mask]

    if "select" in pipeline.named_steps:
        select_mask = pipeline.named_steps["select"].get_support()
        columns = columns[select_mask]

    return list(columns)

selected_features = extract_selected_feature_names(best_model, X_train.columns)
selected_feature_df = pd.DataFrame({"Selected_Feature": selected_features})
selected_feature_df.to_csv(OUTPUT_DIR / f"04_selected_features_{best_model_name}.csv", index=False)

print("Number of selected features:", len(selected_features))
selected_feature_df.head(20)

## 13. Permutation feature importance

Permutation importance is model-agnostic and suitable for reporting the most influential descriptors/fingerprints. It is computed on the independent test set.

In [ ]:
perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    n_repeats=30,
    random_state=RANDOM_STATE,
    scoring="roc_auc",
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance_mean": perm.importances_mean,
    "Importance_std": perm.importances_std,
}).sort_values("Importance_mean", ascending=False)

importance_df.to_csv(OUTPUT_DIR / f"04_permutation_importance_{best_model_name}.csv", index=False)

plot_df = importance_df.head(20).sort_values("Importance_mean")
fig, ax = plt.subplots(figsize=(7, 6), dpi=300)
ax.barh(plot_df["Feature"], plot_df["Importance_mean"], xerr=plot_df["Importance_std"])
ax.set_xlabel("Permutation importance decrease in ROC-AUC")
ax.set_title(f"Top 20 Feature Importance: {best_model_name}")
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"04_permutation_importance_{best_model_name}.png", bbox_inches="tight")
plt.show()

importance_df.head(20)

## 14. Optional SHAP interpretation

This section is optional. It can be slow for SVM models, so a sample of the test set is used when necessary. SHAP plots are useful for explaining which fingerprints/descriptors contribute most strongly to the predicted RecA activity class.

In [ ]:
if not HAS_SHAP:
    print("SHAP is not installed. To enable this section, run: pip install shap")
else:
    X_background = X_train.sample(min(50, len(X_train)), random_state=RANDOM_STATE)
    X_explain = X_test.sample(min(100, len(X_test)), random_state=RANDOM_STATE)

    try:
        # TreeExplainer for tree-based final estimator when possible.
        final_estimator = best_model.named_steps.get("model", best_model)
        if best_model_name in ["RandomForest", "ExtraTrees", "XGBoost"]:
            # Transform features up to the model step for tree SHAP.
            Xt_train = X_background.copy()
            Xt_test = X_explain.copy()
            for step_name, transformer in best_model.steps[:-1]:
                Xt_train = transformer.transform(Xt_train)
                Xt_test = transformer.transform(Xt_test)
            selected_names = extract_selected_feature_names(best_model, X_train.columns)
            Xt_test_df = pd.DataFrame(Xt_test, columns=selected_names)
            explainer = shap.TreeExplainer(final_estimator)
            shap_values = explainer.shap_values(Xt_test_df)
            values = shap_values[1] if isinstance(shap_values, list) else shap_values
            shap.summary_plot(values, Xt_test_df, max_display=20, show=False)
        else:
            # Model-agnostic explainer for SVM or other pipelines.
            explainer = shap.KernelExplainer(best_model.predict_proba, X_background)
            shap_values = explainer.shap_values(X_explain, nsamples=100)
            values = shap_values[1] if isinstance(shap_values, list) else shap_values
            shap.summary_plot(values, X_explain, max_display=20, show=False)

        plt.tight_layout()
        plt.savefig(FIGURE_DIR / f"04_shap_summary_{best_model_name}.png", dpi=300, bbox_inches="tight")
        plt.show()
    except Exception as exc:
        print("SHAP interpretation could not be completed:", exc)

## 15. Export final summary

This final cell saves a compact modeling summary for reporting and reproducibility.

In [ ]:
summary = {
    "input_file": str(INPUT_FILE),
    "dataset_shape_original": list(df.shape),
    "feature_matrix_shape": list(X.shape),
    "target_column": target_col,
    "class_distribution": y.value_counts().to_dict(),
    "test_size": TEST_SIZE,
    "random_state": RANDOM_STATE,
    "cross_validation": f"StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={RANDOM_STATE})",
    "models_benchmarked": list(best_models.keys()),
    "best_model": best_model_name,
    "best_model_test_metrics": test_table[test_table["Model"] == best_model_name].iloc[0].to_dict(),
    "output_directory": str(OUTPUT_DIR.resolve()),
}

with open(OUTPUT_DIR / "04_recA_modeling_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

## Manuscript-ready methodological statement

The RecA molecular classification models were developed using PaDEL-derived molecular descriptors/fingerprints. After metadata removal and binary target encoding, the dataset was split into stratified training and independent test subsets using an 80:20 ratio. Preprocessing, low-variance filtering, univariate feature selection, scaling when required, and classifier training were implemented inside scikit-learn pipelines to minimize information leakage. SVM, Random Forest, ExtraTrees, and XGBoost classifiers were optimized using stratified 10-fold cross-validation on the training set. Final performance was assessed on the independent test set using accuracy, precision, recall, F1-score, MCC, and ROC-AUC. Model interpretation was performed using permutation importance and optional SHAP analysis.